# API Teepee : tests d'authentification

Identifiants lus dans l'environnement : `TEEPEE_CLIENT_ID`, `TEEPEE_CLIENT_SECRET`, `TEEPEE_API_KEY` (voir `.env.example`).

État au dernier passage : l'URL OAuth `/oauth/token` répond **404** et `/v2/ExternAPI/token` répond **500**. L'authentification n'était donc pas encore fonctionnelle.

In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport

## Modèle d'appel GET (à adapter)

In [ ]:
url = "https://api.exemple.com/submissions/123"

headers = {
    "Authorization": f"Bearer {os.environ['TEEPEE_API_KEY']}",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print("Erreur :", response.status_code, response.text)

## Essai 1 : OAuth2 client_credentials (échec 404)

In [ ]:
TOKEN_URL = "https://apisafeplace.teepee.fr/oauth/token"

CLIENT_ID = os.environ["TEEPEE_CLIENT_ID"]
CLIENT_SECRET = os.environ["TEEPEE_CLIENT_SECRET"]

payload = {
    "grant_type": "client_credentials"
}

response = requests.post(
    TOKEN_URL,
    data=payload,
    auth=(CLIENT_ID, CLIENT_SECRET)  # BASIC AUTH
)

response.raise_for_status()

token_data = response.json()
access_token = token_data["access_token"]

print("ACCESS TOKEN :", access_token)

## Essai 2 : endpoint ExternAPI (échec 500)

In [ ]:
TOKEN_URL = "https://apisafeplace.teepee.fr/v2/ExternAPI/token"

payload = {
    "account": "16012",  # celui visible dans Swagger
    "clientId": os.environ["TEEPEE_CLIENT_ID"],
    "clientSecret": os.environ["TEEPEE_CLIENT_SECRET"]
}

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(
    TOKEN_URL,
    json=payload,
    headers=headers
)

response.raise_for_status()

token_data = response.json()
access_token = token_data["token"]  # parfois "accessToken"

print("TOKEN :", access_token)